# LLM Function Calling

This notebook introduces **LLM Function Calling**.

We will reuse the same Azure OpenAI calling style as the reference notebook, but focus on one topic only: how a model can decide to call your Python functions.

## What you will learn

By the end of this notebook, you should be able to:

1. Explain what function calling is.
2. Define a tool schema for the model.
3. See how the model returns a tool call request.
4. Execute the requested Python function yourself.
5. Send the tool result back to the model.
6. Build a small multi-tool workflow.

Beginner note: the model does **not** run your Python code by itself. It only tells you **which function to call** and **what arguments to use**.

In [2]:
# --- Notebook walkthrough: environment ---
# Confirm the working directory (local project folder vs Colab `/content`) before relying on uploads or relative paths.

import os

print("This notebook is designed for Jupyter or Google Colab.")

This notebook is designed for Jupyter or Google Colab.


## Setup

We first install the OpenAI package and create an Azure OpenAI client.

This part follows the same API style used in `07_llm functional prototype.ipynb`.

In [3]:
# --- Notebook walkthrough: install OpenAI SDK ---
# `%pip` installs into the *current* kernel so `from openai import AzureOpenAI` succeeds in the next cells.

%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


Add `HKUST_Azure_API` in the Colab **Secrets** panel, or set it as an environment variable.

If neither is available, the notebook will ask you to paste the key manually.

In [ ]:
# --- Notebook walkthrough: Azure OpenAI client ---
# Never hard-code API keys: Colab Secrets (`userdata`), then `os.environ`, then interactive `getpass`.
# The client needs your resource endpoint, key, and an API version string supported by HKUST's gateway.

import json
import os
from getpass import getpass
from openai import AzureOpenAI

# Colab exposes Secrets via `userdata`; local Jupyter lacks this module—fall back to env / prompt.
try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_api_key(secret_name="HKUST_Azure_API"):
    """Resolve the API key: Colab Secret → environment variable → masked terminal prompt."""
    # 1) Google Colab: Secrets store (recommended for in-class demos on Colab).
    if userdata is not None:
        try:
            value = userdata.get(secret_name)
            if value:
                return value
        except Exception:
            pass

    # 2) Shell / CI: `export HKUST_Azure_API=...` before starting Jupyter.
    value = os.getenv(secret_name)
    if value:
        return value

    # 3) Last resort: type the key once without echo (still avoid committing it to the notebook file).
    return getpass(f"Enter {secret_name}: ")


secret_value_0 = get_api_key("HKUST_Azure_API")

# One shared client object for every `chat.completions.create` call below.
client = AzureOpenAI(
    azure_endpoint="https://hkust.azure-api.net",
    api_key=secret_value_0,
    api_version="2025-02-01-preview",
)

## Helper functions

We will create two helper functions:

- `chat_response(...)` returns the full API response.
- `chat_text(...)` returns only the final text content.

We keep them small so the later examples are easier to read.

In [5]:
# --- Notebook walkthrough: chat API helpers ---
# `chat_response` returns the full completion (so we can read `message.tool_calls`). `chat_text` returns assistant text only.
# Pass `tools` / `tool_choice` when you want function-calling behavior; omit them for plain chat.
# `_normalize_temperature` reflects common gpt-5 deployment rules on this gateway (temperature clamped to 1).

def _normalize_temperature(model_name, temperature):
    # The gpt-5 family on this gateway only supports temperature=1.
    if model_name and str(model_name).startswith("gpt-5"):
        return 1
    return temperature


def chat_response(messages, model_name="gpt-5-mini", temperature=1, max_response_tokens=2000, tools=None, tool_choice=None):
    safe_temperature = _normalize_temperature(model_name, temperature)
    # Build the request body incrementally so optional tool fields stay absent when not needed.
    request = {
        "model": model_name,
        "messages": messages,
        "temperature": safe_temperature,
        "max_tokens": max_response_tokens,
    }
    if tools is not None:
        request["tools"] = tools  # each entry is a `{"type":"function","function":{...}}` schema
    if tool_choice is not None:
        request["tool_choice"] = tool_choice  # e.g. `"none"`, `"auto"`, or a forced `{type,function}` dict

    return client.chat.completions.create(**request)


def chat_text(messages, model_name="gpt-5-mini", temperature=1, max_response_tokens=2000, tools=None, tool_choice=None):
    # When you only need a string answer (warm-up cells), skip inspecting `tool_calls` on the message object.
    response = chat_response(
        messages=messages,
        model_name=model_name,
        temperature=temperature,
        max_response_tokens=max_response_tokens,
        tools=tools,
        tool_choice=tool_choice,
    )
    return response.choices[0].message.content

## Warm-up: a normal chat request

Before using tools, let us confirm that a standard chat request works as expected.

In [6]:
# --- Notebook walkthrough: baseline chat (no tools) ---
# Smoke test the deployment with a normal message list before attaching tool schemas.

# Standard chat roles: system steers behavior; user carries the actual question.
messages = [
    {"role": "system", "content": "You are a helpful assistant for beginners learning LLMs."},
    {"role": "user", "content": "Explain function calling in one short paragraph."},
]

response_text = chat_text(messages)
print(response_text)

Function calling is a way to let a language model produce a structured, machine-readable instruction (usually JSON) that maps directly to a predefined external function or API, so the surrounding application can invoke that function with the model-provided arguments, run the real-world operation (like fetching data, running code, or controlling a device), and return the result to the model; this makes outputs predictable, easier to validate, and lets the model safely interact with tools and systems while keeping natural language for reasoning and planning.


## What is Function Calling?

Function calling lets the model decide that it needs external help.

For example, if the user asks about weather, the model should not guess. Instead, it can request a tool such as `get_current_weather(...)`.

Important idea:

- The model chooses the tool and arguments.
- Your code executes the real Python function.
- Your code sends the tool result back to the model.
- The model then writes the final answer for the user.

## Step 1: define a tool schema

A tool schema is a JSON-like description of a function.

It tells the model:

- the function name
- what the function does
- what arguments are allowed

We will start with one simple tool: `get_current_weather`.

In [7]:
# --- Notebook walkthrough: define a tool schema ---
# JSON-schema-style dict: `name`, natural-language `description`, and typed `parameters` the model must populate.
# This is only a contract for the model; Python implementation is defined separately below.

# `additionalProperties: False` nudges the model to stick to declared fields only.
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current weather for a city. Use this tool when the user asks about current weather conditions.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city name, for example Hong Kong, Tokyo, or London."
                }
            },
            "required": ["city"],
            "additionalProperties": False
        }
    }
}

# The API accepts a *list* of tools even when you only register one.
tools = [weather_tool]
print(json.dumps(weather_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_current_weather",
    "description": "Get the current weather for a city. Use this tool when the user asks about current weather conditions.",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "The city name, for example Hong Kong, Tokyo, or London."
        }
      },
      "required": [
        "city"
      ],
      "additionalProperties": false
    }
  }
}


Beginner note: the schema does **not** contain the Python implementation yet.

At this stage, we are only telling the model what tool is available.

## Step 2: let the model decide whether to call the tool

Now we send a weather question and include the tool schema.

If the model thinks the tool is useful, it may return a `tool_calls` field instead of a normal text answer.

In [8]:
# --- Notebook walkthrough: model chooses a tool ---
# With `tools=[...]`, the assistant may return `tool_calls` (structured intent) instead of answering from memory.

# System text steers the model toward tool use instead of inventing live conditions.
messages = [
    {
        "role": "system",
        "content": "You are a weather assistant. When the user asks about current weather, use the provided tool instead of guessing."
    },
    {
        "role": "user",
        "content": "What is the weather like in Hong Kong right now?"
    },
]

# Passing the same `tools` list advertises `get_current_weather` for this completion.
response = chat_response(messages, tools=tools)
assistant_message = response.choices[0].message
print(assistant_message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HJkWvrbGRvKF0S3JpFMH4Gdv', function=Function(arguments='{"city":"Hong Kong"}', name='get_current_weather'), type='function')])


## Inspect the tool call request

The returned object usually contains:

- the tool name
- a JSON string of arguments
- a tool call ID

This is the model saying: "Please run this function for me."

In [9]:
# --- Notebook walkthrough: inspect tool_calls ---
# Each entry includes `id` (echoed in the later `tool` message), `function.name`, and `function.arguments` as JSON text.

# `None` or empty means the model answered directly without requesting tools.
tool_calls = assistant_message.tool_calls

if tool_calls:
    for tool_call in tool_calls:
        print("Tool call ID:", tool_call.id)
        print("Function name:", tool_call.function.name)
        print("Arguments:", tool_call.function.arguments)
        print("-" * 40)
else:
    print("The model did not request any tool call.")

Tool call ID: call_HJkWvrbGRvKF0S3JpFMH4Gdv
Function name: get_current_weather
Arguments: {"city":"Hong Kong"}
----------------------------------------


## Step 3: implement the Python function

Next, we write the real Python function.

To keep the notebook simple and reproducible, we will use **mock data** instead of a real weather API.

That is enough to learn the function-calling workflow.

In [10]:
# --- Notebook walkthrough: implement the tool in Python ---
# Mock lookup table keeps demos reproducible; replace the body with a real API client when you ship.

def get_current_weather(city):
    # Tiny in-memory table; keys are normalized to lowercase for robust matching.
    sample_weather = {
        "hong kong": {"temperature_celsius": 24, "condition": "humid with sunny periods", "humidity": "78%"},
        "tokyo": {"temperature_celsius": 16, "condition": "cool and clear", "humidity": "52%"},
        "london": {"temperature_celsius": 11, "condition": "cloudy with light wind", "humidity": "70%"},
    }

    city_key = city.strip().lower()
    weather = sample_weather.get(
        city_key,
        {"temperature_celsius": 22, "condition": "weather data not found, using demo fallback", "humidity": "unknown"},
    )

    # Everything here must be JSON-serializable—it becomes `tool` message `content`.
    return {
        "city": city,
        "temperature_celsius": weather["temperature_celsius"],
        "condition": weather["condition"],
        "humidity": weather["humidity"],
        "source": "mock data for classroom demonstration"
    }

## Step 4: manually execute the tool call

Here we read the arguments from the model, call the Python function, and print the result.

This manual step is useful because it makes the whole process very concrete.

In [11]:
# --- Notebook walkthrough: manual tool execution ---
# Parse JSON arguments, call the matching Python function with `**kwargs`, and preview the payload you will return to the API.

if assistant_message.tool_calls:
    first_tool_call = assistant_message.tool_calls[0]
    # `arguments` is a JSON string chosen by the model; validate in production before `**` unpacking.
    tool_arguments = json.loads(first_tool_call.function.arguments)
    print(tool_arguments)
    tool_result = get_current_weather(**tool_arguments)
    print(json.dumps(tool_result, indent=2))
else:
    print("No tool call was returned, so there is nothing to execute.")

{'city': 'Hong Kong'}
{
  "city": "Hong Kong",
  "temperature_celsius": 24,
  "condition": "humid with sunny periods",
  "humidity": "78%",
  "source": "mock data for classroom demonstration"
}


## Step 5: send the tool result back to the model

After running the Python function, we append two things to the conversation:

1. The assistant message that contains the tool request.
2. A `tool` message that contains the tool output.

Then we call the model again so it can write the final user-facing answer.

In [14]:
# --- Notebook walkthrough: two-round chat pattern ---
# Round 1: assistant may emit tool calls. Append that assistant message, append one `tool` message per call (with matching `tool_call_id`).
# Round 2: call the API again so the model writes the final natural-language answer using tool output.

def run_weather_assistant(user_input, model_name="gpt-5-mini"):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful weather assistant. Use the tool when weather data is needed. Explain the result in simple English."
        },
        {"role": "user", "content": user_input},
    ]

    # First model turn: may produce tool calls instead of a final natural-language answer.
    first_response = chat_response(messages, model_name=model_name, tools=[weather_tool])
    assistant_message = first_response.choices[0].message

    if not assistant_message.tool_calls:
        return assistant_message.content

    # Must preserve tool_call ids so the follow-up `tool` rows correlate correctly.
    messages.append(assistant_message.model_dump(exclude_none=True))

    for tool_call in assistant_message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        result = get_current_weather(**arguments)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "content": json.dumps(result),
            }
        )

    # Second model turn: consumes tool outputs and should return user-facing prose.
    final_response = chat_response(messages, model_name=model_name, tools=[weather_tool])
    return final_response.choices[0].message.content

In [13]:
# --- Notebook walkthrough: end-to-end weather assistant ---
# One line for students; internally the helper may perform multiple model rounds and tool executions.

# Student-facing API: hide the two-step `chat` + `tool` choreography inside one helper call.
final_answer = run_weather_assistant("I will visit Hong Kong this afternoon. What is the weather like?")
print(final_answer)

This afternoon in Hong Kong: 24°C (about 75°F), humid with sunny periods, humidity 78%.

What that means in simple terms
- Expect warm, muggy air with sun and some breaks of cloud — not heavy rain right now, but it will feel a bit sticky because of the high humidity.

Practical tips
- Wear light, breathable clothing and comfortable shoes.
- Bring water and stay hydrated.
- Use sunscreen, sunglasses and a hat for the sun.
- Carry a small umbrella or light rain jacket only if you want extra protection — no heavy storm is indicated, but conditions can change.

I recommend checking again just before you leave in case conditions change.


## Optional control: disable tool calls

Sometimes you want to prevent the model from using tools.

You can do that with `tool_choice="none"`.

This is useful for debugging or for comparing behavior with and without tools.

In [15]:
# --- Notebook walkthrough: disable tools (`tool_choice`) ---
# `tool_choice="none"` forces a text reply even if tools are listed—handy for debugging or policy toggles.

messages = [
    {
        "role": "system",
        "content": "You are a weather assistant. If you cannot use a tool, answer carefully and mention the limitation."
    },
    {
        "role": "user",
        "content": "What is the weather like in Tokyo right now?"
    },
]

# Tools are still *declared*, but the model is not allowed to emit `tool_calls` for this request.
response_no_tools = chat_response(messages, tools=[weather_tool], tool_choice="none")
print(response_no_tools.choices[0].message.content)

I can’t fetch live weather data at the moment, so I can’t give the exact current conditions in Tokyo. I can either (A) give a quick, up-to-date-check instruction so you can see it now, or (B) describe the typical weather for this time of year and what to expect — which would be an estimate, not real-time.

Quick summary for late March in Tokyo (typical conditions):
- Temperature: daytime highs often around 12–18°C (54–64°F); nights and early mornings around 4–10°C (39–50°F).
- Precipitation: spring can be changeable — light rain showers are possible but not constant.
- Wind/humidity: generally light to moderate breeze and moderate humidity.
- Other: cherry-blossom season usually begins late March to early April in central Tokyo, so days can be mild and pleasant but mornings/evenings still cool.

Practical tips:
- Wear layers (light jacket or sweater) so you can adjust for cooler mornings and milder afternoons.
- Carry a small umbrella or water-resistant layer in case of showers.

If yo

## Optional control: force a specific tool

You can also force the model to call a given function.

This is useful when you already know that a tool must be used.

In [16]:
# --- Notebook walkthrough: force a specific tool ---
# `tool_choice` can name one function so the model must emit that call (e.g., when your app already decided the action).

# Reuses the `messages` list from the previous cell (Tokyo question) to highlight `tool_choice` effects only.
forced_response = chat_response(
    messages=messages,
    tools=[weather_tool],
    tool_choice={"type": "function", "function": {"name": "get_current_weather"}},
)

forced_message = forced_response.choices[0].message
print(forced_message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LVdAnBhJ8t6GrdwyELU03oJI', function=Function(arguments='{"city":"Tokyo"}', name='get_current_weather'), type='function')])


## Add a second tool

Now let us make the example a little more interesting.

We will add another function called `convert_temperature`.

This lets the model combine tools in a small workflow:

1. Get the weather in Celsius.
2. Convert the temperature if the user asks for Fahrenheit.

In [17]:
# --- Notebook walkthrough: multiple tools in one list ---
# The model picks among tools using names, descriptions, and the user question; keep schemas distinct and concise.

convert_temperature_tool = {
    "type": "function",
    "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value from one unit to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "The numeric temperature value to convert."
                },
                "from_unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                },
                "to_unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["value", "from_unit", "to_unit"],
            "additionalProperties": False
        }
    }
}

# Order is not semantically important; descriptions help the model disambiguate which tool to pick.
all_tools = [weather_tool, convert_temperature_tool]
print("Number of tools:", len(all_tools))

Number of tools: 2


## Build a simple tool router

When the model returns a tool call, our code needs to:

- read the function name
- find the matching Python function
- execute it with the provided arguments
- package the result as a `tool` message

This pattern is common in real applications.

In [18]:
# --- Notebook walkthrough: router + multi-step tool loop ---
# `TOOL_FUNCTIONS` maps API-exposed names to real Python callables. `run_tool_loop` repeats until the assistant stops requesting tools.

def convert_temperature(value, from_unit, to_unit):
    # Pure deterministic math—ideal as a tool so the LLM does not approximate conversions.
    if from_unit == to_unit:
        converted_value = value
    elif from_unit == "celsius" and to_unit == "fahrenheit":
        converted_value = (value * 9 / 5) + 32
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        converted_value = (value - 32) * 5 / 9
    else:
        raise ValueError("Unsupported temperature conversion")

    return {
        "original_value": value,
        "from_unit": from_unit,
        "converted_value": round(converted_value, 2),
        "to_unit": to_unit,
    }


TOOL_FUNCTIONS = {
    "get_current_weather": get_current_weather,
    "convert_temperature": convert_temperature,
}


def execute_tool_call(tool_call):
    # Dispatch table: model-facing string names must match keys exactly.
    function_name = tool_call.function.name
    function_arguments = json.loads(tool_call.function.arguments)
    function_to_run = TOOL_FUNCTIONS[function_name]
    function_result = function_to_run(**function_arguments)

    # Shape required by the Chat Completions API for tool outputs.
    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": function_name,
        "content": json.dumps(function_result),
    }


def run_tool_loop(user_input, tools, system_prompt, model_name="gpt-5-mini"):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]

    while True:
        response = chat_response(messages, model_name=model_name, tools=tools)
        assistant_message = response.choices[0].message

        if assistant_message.tool_calls:
            messages.append(assistant_message.model_dump(exclude_none=True))
            for tool_call in assistant_message.tool_calls:
                tool_message = execute_tool_call(tool_call)
                messages.append(tool_message)
            continue  # Ask the model again now that tool outputs exist.

        # No pending tools: `content` should hold the natural-language answer.
        return assistant_message.content, messages

## Multi-tool example

In this example, the model should first fetch the weather, then convert the Celsius value to Fahrenheit.

We will also give the system prompt a clear rule: use the conversion tool instead of doing the math directly.

In [19]:
# --- Notebook walkthrough: chained tools demo ---
# Typical pattern: fetch weather, then call `convert_temperature` when the user wants another unit—prompt rules steer tool use.

system_prompt = """
You are a helpful travel assistant.
Use tools when the user asks for weather data.
If a temperature unit conversion is needed, use the convert_temperature tool instead of doing the math yourself.
Explain the final result in clear and simple English.
"""

# `conversation` captures every turn for debugging; `final_answer` is the string shown to the user.
final_answer, conversation = run_tool_loop(
    user_input="What is the weather in Hong Kong right now? Please also give me the temperature in Fahrenheit.",
    tools=all_tools,
    system_prompt=system_prompt,
)

print(final_answer)

Right now in Hong Kong it’s 24°C (75.2°F) and humid with sunny periods, with about 78% humidity.


In [20]:
# --- Notebook walkthrough: print full message transcript ---
# Walk `conversation` to show roles in API order: `system`, `user`, `assistant` (with `tool_calls`), `tool`, then final `assistant` text.

for message in conversation:
    print(message["role"].upper())
    if message["role"] == "assistant" and "tool_calls" in message:
        # Pretty-print structured tool requests; plain-text replies use the branch below.
        print(json.dumps(message, indent=2))
    else:
        print(message.get("content"))
    print("=" * 60)

SYSTEM

You are a helpful travel assistant.
Use tools when the user asks for weather data.
If a temperature unit conversion is needed, use the convert_temperature tool instead of doing the math yourself.
Explain the final result in clear and simple English.

USER
What is the weather in Hong Kong right now? Please also give me the temperature in Fahrenheit.
ASSISTANT
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_SEmvddxXoXZ8LRQPt3WIY43r",
      "function": {
        "arguments": "{\"city\":\"Hong Kong\"}",
        "name": "get_current_weather"
      },
      "type": "function"
    }
  ]
}
TOOL
{"city": "Hong Kong", "temperature_celsius": 24, "condition": "humid with sunny periods", "humidity": "78%", "source": "mock data for classroom demonstration"}
ASSISTANT
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "call_91vWTMzj7pEL5yrHdyFBE7dC",
      "function": {
        "arguments": "{\"value\":24,\"from_unit\":\"celsius\",\"to_unit\":\"fahrenheit\"}",
   

## Why function calling is useful

Function calling is helpful when the model needs:

- fresh data from an external API
- precise calculations
- database lookups
- actions in another system

Typical examples include weather bots, order tracking assistants, database query tools, scheduling agents, and internal company copilots.

## Good practices

When you design tools for a real application, try to follow these rules:

- Keep function names clear and specific.
- Write descriptions that help the model choose the right tool.
- Keep parameter schemas simple.
- Validate tool arguments in Python before executing risky actions.
- Do not let the model directly perform unsafe operations.
- Log tool calls for debugging.

Beginner note: function calling improves structure, but it does not replace careful engineering.

## Summary

You have now seen the full workflow:

1. Define a tool schema.
2. Send the tool schema with the user message.
3. Let the model request a tool call.
4. Execute the Python function yourself.
5. Return the tool result to the model.
6. Get the final answer.

This is the core pattern behind many practical LLM applications.

## Practice

Try these small extensions:

1. Add a new tool called `get_air_quality`.
2. Build a simple `get_course_info` tool for a university assistant.
3. Add argument validation for missing or invalid city names.
4. Save tool call logs in a Python list for later inspection.

These exercises will help you move from a toy demo to a more realistic agent workflow.